# 00 — Quick Start

Build a cassette-backed hypergraph reasoner from documents in five minutes. No training, no GPU, no model downloads — define your schema and go. End-to-end runs in **under five seconds** on a laptop CPU.

```
pip install -e ./
```

Runtime: `torch`, `transformers`, `numpy`, `pyarrow`, `fsspec`. Optional: `s3fs` (for `s3://` URIs), `strands-agents` (for the `Analyst`).

What you'll build:

1. A typed schema that projects text into your concept vocabulary.
2. An `InfonStore` — cassettes on local disk (or S3 via URI swap).
3. Ingest 5 docs; read the extraction diagnostic.
4. Ask a single-claim question; get a calibrated verdict.
5. Ask a multi-hop question via `connect()`.
6. Edit the schema and migrate in milliseconds via `SchemaFunctor`.

## 1. Define your schema

A schema tells InfonEngine *what to look for*. Each **anchor** is a typed vocabulary entry with token strings that project onto the SPLADE vocabulary. Four canonical types:

| Type | Role in triple | Example |
|---|---|---|
| `actor` | subject (and object, post-fix) | toyota, tesla, catl |
| `relation` | predicate | invest, partner, acquire |
| `feature` | object | battery, ev, solid_state |
| `market` / `location` | object | japan, north_america, china |

The schema is load-bearing: the reasoner only reasons over anchors you define. Missing a concept = silently missing extractions. `store.extraction_report()` tells you when that happens.

In [ ]:
import json, tempfile, os

SCHEMA = {
    # Actors — eligible as subject AND object (dual-partition)
    "toyota":    {"type": "actor", "tokens": ["toyota"]},
    "honda":     {"type": "actor", "tokens": ["honda"]},
    "tesla":     {"type": "actor", "tokens": ["tesla"]},
    "panasonic": {"type": "actor", "tokens": ["panasonic"]},
    "catl":      {"type": "actor", "tokens": ["catl"]},

    # Relations — predicate role
    "invest":  {"type": "relation", "tokens": ["invest", "invested", "investment"]},
    "partner": {"type": "relation", "tokens": ["partner", "partners", "partnered", "partnership"]},
    "supply":  {"type": "relation", "tokens": ["supply", "supplies", "supplier", "sources"]},

    # Features — object role
    "batteries":   {"type": "feature", "tokens": ["battery", "batteries"]},
    "solid_state": {"type": "feature", "tokens": ["solid-state", "solid state"]},
    "ev":          {"type": "feature", "tokens": ["ev", "electric vehicle"]},

    # Markets — where
    "japan":         {"type": "market", "tokens": ["japan"]},
    "north_america": {"type": "market", "tokens": ["north america", "us", "united states"]},
}

tmpdir = tempfile.mkdtemp(prefix="infon_qs_")
schema_path = os.path.join(tmpdir, "schema.json")
with open(schema_path, "w") as f:
    json.dump(SCHEMA, f)
print(f"Schema: {len(SCHEMA)} anchors")

## 2. Create the InfonStore

One import, one root URI, one object. Local path now, S3 later — the API is the same.

```python
store = InfonStore("./data/chips", schema_path="schema.json")         # local
store = InfonStore("s3://acme/chips", schema_path="schema.json")      # S3
```

The store creates `cassettes/`, `index/`, `docs/`, and `_manifest/` under the root on first ingest. Nothing else lives in your process.

In [ ]:
from infon.cassette import InfonStore, Query

store_root = os.path.join(tmpdir, "store")
store = InfonStore(store_root, schema_path=schema_path)
print(store)

## 3. Ingest documents

Each document is `{"id": ..., "text": ..., "timestamp": ...}`. `ingest()` returns a summary **and an extraction report** that flags coverage issues — docs that produced zero infons, anchors that never fired, overfit objects. Read the report before you trust the queries.

Re-ingesting the same document is a no-op: cassette IDs are content-addressed over `text + schema`, so the second call short-circuits.

In [ ]:
documents = [
    {"id": "d1", "text": "Toyota invests in solid-state battery technology in Japan.",
     "timestamp": "2026-01-05"},
    {"id": "d2", "text": "Toyota partnered with Panasonic on battery supply.",
     "timestamp": "2026-01-12"},
    {"id": "d3", "text": "Panasonic sources battery cells from CATL.",
     "timestamp": "2026-01-20"},
    {"id": "d4", "text": "Tesla produces batteries at its North America factory.",
     "timestamp": "2026-02-05"},
    {"id": "d5", "text": "Honda partners with CATL on battery supply.",
     "timestamp": "2026-02-12"},
]

result = store.ingest(documents)
print(f"ingested: {len(result['ingested'])}  skipped: {len(result['skipped'])}  "
      f"errors: {len(result['errors'])}  n_infons: {result['n_infons']}")
print()
print(result["report"].summary())

## 4. Inspect what got extracted

Before querying, eyeball the triples. If important sentences produced zero infons, the schema needs more tokens — `extraction_report` will have told you already.

Every hit is a pointer into a cassette; we haven't hydrated any sentences yet. The DSL runs entirely in index space.

In [ ]:
for h in Query().run(store.manifest):
    pol = "¬" if h.loc.polarity == 0 else " "
    print(f"  {pol}{h.loc.subject:>10} {h.loc.predicate:<8} {h.loc.object:<14} "
          f"conf={h.loc.confidence:.2f}  t={h.loc.timestamp}")

## 5. Ask a single-claim question

`store.ask()` takes a `Query` that pins the triple you want to verify. The reasoner retrieves matching infons, computes per-infon Dempster–Shafer mass functions, and combines them via Dempster's rule.

Every verdict carries `(supports, refutes, theta)`. **θ is the headline number**: residual ignorance. When the system has evidence it commits; when it doesn't, θ stays high.

In [ ]:
v = store.ask(Query().where(subject="toyota", predicate="invest",
                             object="solid_state"))
print(f"verdict:  {v.label}")
print(f"supports: {v.mass.supports:.3f}")
print(f"refutes:  {v.mass.refutes:.3f}")
print(f"theta:    {v.mass.theta:.3f}")
print()
print("sources:")
for s in v.sources[:3]:
    print(f"  \u2022 {s.sentence}   (conf={s.confidence:.2f}, doc={s.doc_id})")

### θ on claims the corpus doesn't speak to

The litmus test for honesty: on NEI (*Not Enough Info*) claims, θ should approach 1.0. The manifest pruner short-circuits before any hydration — 0 range gets, pure index scan, θ=1.0.

In [ ]:
v = store.ask(Query().where(subject="tesla", predicate="acquire",
                             object="catl"))
print(f"verdict: {v.label}")
print(f"theta:   {v.mass.theta:.3f}   (near 1.0 = honest unknown)")
print(f"gets:    {v.range_gets}        (0 = no hydration, pruner saved us)")

## 6. Ask a multi-hop question

`store.connect(source, target)` runs an MCTS search over the hypergraph, following connective edges (partner/supply/acquire/license). Chain-aware: a retracted edge at any hop turns SUPPORTS into REFUTES. Returns the full path as `sources`.

When `<root>/_model/gnn.pt` exists (see module 12), the trained sheaf GNN re-scores the MCTS chain as a prior. For this quick start we run symbolic-only — it still solves clean chains trivially.

In [ ]:
v = store.connect("toyota", "catl")   # chain: toyota \u2192 panasonic \u2192 catl
print(f"verdict:  {v.label}")
print(f"supports: {v.mass.supports:.3f}")
print(f"theta:    {v.mass.theta:.3f}")
print(f"chain:")
for s in v.sources:
    print(f"  {s.subject} \u2192 {s.predicate} \u2192 {s.object}   ({s.sentence})")

## 7. One tree walk, many targets

`store.any_of(source, targets)` resolves connectivity to every target in a single MCTS walk. Cost is roughly `O(1)` in the number of targets — a chain that passes through an intermediate resolves that intermediate for free.

In [ ]:
vs = store.any_of("toyota", {"catl", "panasonic", "honda", "tesla"})
for target, v in sorted(vs.items(), key=lambda kv: -kv[1].mass.supports):
    print(f"  {target:<12} {v.label:<18}  S={v.mass.supports:.2f}  \u03b8={v.mass.theta:.2f}")

## 8. Schema migration without reingestion

Users edit schemas. Traditional stores force re-extraction; cassettes let you push the old infons through a functor and write new cassettes tagged with the new schema. Old cassettes stay queryable at prior snapshots.

Here we merge `panasonic` and `catl` into a single `catl_corp` anchor and delete `ev`. Preview first, then commit.

In [ ]:
from infon.cassette import SchemaFunctor

SCHEMA_V2 = dict(SCHEMA)
SCHEMA_V2["catl_corp"] = {"type": "actor",
                           "tokens": ["catl", "panasonic", "catl_corp"]}
for name in ("catl", "panasonic", "ev"):
    SCHEMA_V2.pop(name, None)

schema_v2_path = os.path.join(tmpdir, "schema_v2.json")
with open(schema_v2_path, "w") as f:
    json.dump(SCHEMA_V2, f)

functor = SchemaFunctor(
    merge={"catl": "catl_corp", "panasonic": "catl_corp"},
    delete={"ev"},
)

print("\u2500\u2500 preview (no writes) \u2500\u2500")
print(store.plan_migration(functor, schema_v2_path).summary())

In [ ]:
import time
t0 = time.perf_counter()
report = store.migrate(functor, schema_v2_path)
wall = (time.perf_counter() - t0) * 1000
print(f"migrated in {wall:.0f}ms  (compare to ingest: seconds)")

# toyota/partner/panasonic should now be toyota/partner/catl_corp
hits = Query().where(subject="toyota", predicate="partner",
                     object="catl_corp").run(store.manifest)
print(f"toyota/partner/catl_corp  hits={len(hits)}")

## What just happened

1. Defined a 13-anchor typed schema.
2. Created an `InfonStore` backed by cassettes (swap local path for `s3://` and the API is identical).
3. Ingested 5 documents → SPLADE encoded → extracted → wrote immutable cassettes + Parquet indexes.
4. Read the extraction report — coverage diagnostics for free.
5. Asked a single-claim question → calibrated Dempster–Shafer verdict with cited sources.
6. Asked a claim the corpus doesn't answer → θ → 1.0, 0 range gets (pruner short-circuits).
7. Asked a multi-hop connectivity question → MCTS found `toyota → panasonic → catl`.
8. Migrated the schema via functor → **milliseconds, not seconds** — old cassettes stayed, new ones joined the manifest.

**No training data, no GPU, no model download beyond the 17 MB SPLADE weights.** Cassettes sit on any fsspec-addressable filesystem (local, S3, GCS, Azure).

### What's next

- **[03 — Querying](03_querying.ipynb)**: the full DSL (grammar, timeline, logic, trajectory, constraint, hierarchy expansion).
- **[06 — Agent Tools](06_agent_tools.ipynb)**: the Strands-powered `Analyst` with 9 tools — conversational entrypoint.
- **[08 — Category Theory](08_category_theory.ipynb)**: sheaf coherence + the shipped Kan migration.
- **12 — Sheaf GNN** (new): synthgen-trained reasoner prior that ships with every store.
- **13 — MCTS**: polarity-aware chain mass, connective-predicate inference, GNN-augmented search.

In [ ]:
import shutil
shutil.rmtree(tmpdir)
print("Done.")